# Naive Bayes

### Overview

- [Setup and Data Loading](#Setup-and-Data-Loading)
- [Data Understanding](#Data-Understanding)
- [Encoding the Features](#Encoding-the-Features)
- [The Majority Baseline](#The-Majority-Baseline)
- [A First Naive Bayes Model](#A-First-Naive-Bayes-Model)
- [Tuning Naive Bayes](#Tuning-Naive-Bayes)
- [Final Evaluation and the Imbalance Effect](#Final-Evaluation-and-the-Imbalance-Effect)
- [Saving the Results](#Saving-the-Results)

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report

## Setup and Data Loading

The train/test split and the leakage-safe `publisher_tier` are already created in Phase B.

In [2]:
SCOPE = "scope_2000_2018"
DATA_DIR = "data/processed/" + SCOPE

X_train = pd.read_csv(DATA_DIR + "/X_train.csv")
X_test  = pd.read_csv(DATA_DIR + "/X_test.csv")
y_train = pd.read_csv(DATA_DIR + "/y_train.csv")["Hit"]
y_test  = pd.read_csv(DATA_DIR + "/y_test.csv")["Hit"]

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
X_train.head()

X_train shape: (13681, 3)
X_test shape:  (3421, 3)


,genre,platform_family,publisher_tier
0,Shooter,PC,2
1,Sports,Xbox,2
2,Strategy,Nintendo,2
3,Action,PlayStation,0
4,Fighting,Xbox,0


## Data Understanding

The model only gets the three pre-release features from Phase B.

For Naive Bayes the class prior is important, so I check the class balance first.

In [3]:
print('Hit  (1):', len(y_train[y_train == 1]) / len(y_train))
print('Flop (0):', len(y_train[y_train == 0]) / len(y_train))

print()
print('Training labels:')
print(y_train.value_counts().sort_index())

Hit  (1): 0.2034938966449821
Flop (0): 0.7965061033550179

Training labels:
Hit
0    10897
1     2784
Name: count, dtype: int64


## Encoding the Features

`CategoricalNB` cannot directly use strings. Like in the Naive Bayes lecture notebook, I use `OrdinalEncoder` to convert categories to integer IDs.

For this model the numbers are only category IDs. They are not distances.

In [4]:
X_train_cat = X_train.astype(str)
X_test_cat = X_test.astype(str)

for col in X_train_cat.columns:
    unseen = sorted(set(X_test_cat[col]) - set(X_train_cat[col]))
    print(col, 'unseen test categories:', unseen)

enc = OrdinalEncoder()
X_train_enc = enc.fit_transform(X_train_cat)
X_test_enc = enc.transform(X_test_cat)

print('Encoded training shape:', X_train_enc.shape)
X_train_enc[:5]

genre unseen test categories: []
platform_family unseen test categories: []
publisher_tier unseen test categories: []
Encoded training shape: (13681, 3)


array([[10.,  2.,  2.],
       [12.,  4.,  2.],
       [13.,  0.,  2.],
       [ 0.,  3.,  0.],
       [ 3.,  4.,  0.]])

## The Majority Baseline

A simple baseline always predicts the majority class, which is `Flop`.

In [5]:
baseline_pred = np.zeros(len(y_test), dtype=int)

print('Baseline accuracy:', accuracy_score(y_test, baseline_pred))
print('Baseline F1      :', f1_score(y_test, baseline_pred, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, baseline_pred))

Baseline accuracy: 0.7965507161648641
Baseline F1      : 0.0
Confusion matrix:
[[2725    0]
 [ 696    0]]


## A First Naive Bayes Model

First I train a basic categorical Naive Bayes model with the default Laplace smoothing.

In [6]:
nb_first = CategoricalNB()
nb_first.fit(X_train_enc, y_train)

pred = nb_first.predict(X_test_enc)
proba = nb_first.predict_proba(X_test_enc)[:, 1]

print('Test accuracy:', accuracy_score(y_test, pred))
print('Test F1      :', f1_score(y_test, pred))
print('Test ROC-AUC :', roc_auc_score(y_test, proba))
print('Confusion matrix:')
print(confusion_matrix(y_test, pred))

Test accuracy: 0.797427652733119
Test F1      : 0.2376237623762376
Test ROC-AUC : 0.7744503321733629
Confusion matrix:
[[2620  105]
 [ 588  108]]


## Tuning Naive Bayes

The lecture explains Laplace smoothing. In scikit-learn this is controlled by `alpha`, so I try a few values.

In [7]:
param_grid = {'alpha': [0.1, 0.5, 1.0, 2.0, 5.0]}

nb_model = CategoricalNB()
nb_cv = GridSearchCV(nb_model, param_grid, scoring='roc_auc', cv=5, n_jobs=-1)
nb_cv.fit(X_train_enc, y_train)

print('Best CV ROC-AUC:', nb_cv.best_score_)
print('Best parameters:', nb_cv.best_params_)

Best CV ROC-AUC: 0.7683378870691064
Best parameters: {'alpha': 0.5}


## Final Evaluation and the Imbalance Effect

The final comparison uses the same tuned procedure twice:

1. train on the original imbalanced training set
2. train on a 50/50 down-sampled training set

The test set is not changed.

In [8]:
def evaluate(model, X_te, y_te, model_name, resample):
    proba = model.predict_proba(X_te)[:, 1]
    pred = model.predict(X_te)

    roc = roc_auc_score(y_te, proba)
    f1 = f1_score(y_te, pred)
    acc = accuracy_score(y_te, pred)

    print('--- %s (resample=%s) ---' % (model_name, resample))
    print('ROC-AUC : %.3f' % roc)
    print('F1      : %.3f' % f1)
    print('Accuracy: %.3f' % acc)
    print('Confusion matrix [rows = true 0/1, cols = predicted 0/1]:')
    print(confusion_matrix(y_te, pred))
    print(classification_report(y_te, pred, zero_division=0))

    return {'model': model_name, 'resample': resample,
            'roc_auc': roc, 'f1': f1, 'accuracy': acc}


def fit_nb(X_tr, y_tr):
    gs = GridSearchCV(CategoricalNB(), param_grid,
                      scoring='roc_auc', cv=5, n_jobs=-1, refit=True)
    gs.fit(X_tr, y_tr)
    print('best parameters:', gs.best_params_)
    return gs.best_estimator_

### Run 1 — no resampling

In [9]:
nb_none = fit_nb(X_train_enc, y_train)
row_none = evaluate(nb_none, X_test_enc, y_test, 'naive_bayes', 'none')

best parameters: {'alpha': 0.5}
--- naive_bayes (resample=none) ---
ROC-AUC : 0.774
F1      : 0.238
Accuracy: 0.797
Confusion matrix [rows = true 0/1, cols = predicted 0/1]:
[[2620  105]
 [ 588  108]]
              precision    recall  f1-score   support

           0       0.82      0.96      0.88      2725
           1       0.51      0.16      0.24       696

    accuracy                           0.80      3421
   macro avg       0.66      0.56      0.56      3421
weighted avg       0.75      0.80      0.75      3421



### Run 2 — 50/50 down-sampling

In [10]:
train = pd.DataFrame(X_train_enc)
train['Hit'] = y_train.values

hits = train[train['Hit'] == 1]
flops = train[train['Hit'] == 0]

flops_down = flops.sample(n=len(hits), random_state=0)
train_bal = pd.concat([hits, flops_down]).sample(frac=1, random_state=0)

X_train_bal = train_bal.drop(columns='Hit')
y_train_bal = train_bal['Hit']

print('Balanced training set size:', len(y_train_bal))
print('Hit  (1):', len(y_train_bal[y_train_bal == 1]))
print('Flop (0):', len(y_train_bal[y_train_bal == 0]))

Balanced training set size: 5568
Hit  (1): 2784
Flop (0): 2784


In [11]:
nb_down = fit_nb(X_train_bal, y_train_bal)
row_down = evaluate(nb_down, X_test_enc, y_test, 'naive_bayes', 'downsample')

best parameters: {'alpha': 2.0}
--- naive_bayes (resample=downsample) ---
ROC-AUC : 0.775
F1      : 0.495
Accuracy: 0.665
Confusion matrix [rows = true 0/1, cols = predicted 0/1]:
[[1714 1011]
 [ 134  562]]
              precision    recall  f1-score   support

           0       0.93      0.63      0.75      2725
           1       0.36      0.81      0.50       696

    accuracy                           0.67      3421
   macro avg       0.64      0.72      0.62      3421
weighted avg       0.81      0.67      0.70      3421



## Saving the Results

The two runs are saved with the same columns as the other Phase C model notebooks.

In [12]:
os.makedirs('results', exist_ok=True)

results = pd.DataFrame([row_none, row_down])
results = results[['model', 'resample', 'roc_auc', 'f1', 'accuracy']]

out_path = 'results/naive_bayes.csv'
results.to_csv(out_path, index=False)
print('Saved', out_path)
results

Saved results/naive_bayes.csv


,model,resample,roc_auc,f1,accuracy
0,naive_bayes,none,0.774490,0.237624,0.797428
1,naive_bayes,downsample,0.774917,0.495372,0.665303
